In [2]:
# ==============================================================================
# kNN 解析：ID(X)除去 ＋ OOD分割 ＋ fixed出力版
# ==============================================================================
# --- 必要なパッケージの読み込み ---
library(caret)
library(Metrics)
library(ggplot2)
library(lattice)
library(dplyr) 

# --- OOD分割関数 ---
create_ood_split <- function(df, feature_cols, ood_ratio = 0.1) {
    df_features <- df[, feature_cols, drop = FALSE]
    dist_matrix <- dist(df_features, method = "euclidean")
    avg_distances <- as.matrix(dist_matrix) %>% colMeans()

    num_ood <- floor(nrow(df) * ood_ratio)
    if (num_ood == 0 && nrow(df) > 0) num_ood <- 1 

    ood_indices <- order(avg_distances, decreasing = TRUE)[1:num_ood]
    test_set <- df[ood_indices, ]
    train_set <- df[-ood_indices, ]

    cat(paste0("  OOD Split: Total=", nrow(df), ", Train=", nrow(train_set), ", Test(OOD)=", nrow(test_set), "\n"))
    return(list(train = train_set, test = test_set, ood_indices = ood_indices))
}

# --- 設定 ---
file_names <- c(
    "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
    "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
    "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
    "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)

base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
today <- format(Sys.Date(), "%Y%m%d")

target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_k", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features", "OOD_R2", "OOD_RMSE")
result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

# --- メインループ ---
for (file in file_names) {
    filepath <- paste0(base_path, file)
    if (!file.exists(filepath)) next
    
    cat("\n=== Processing:", file, "===\n")
    df_all <- read.csv(filepath)
    
    # 特徴量抽出
    feature_vars_all <- setdiff(colnames(df_all), target_vars)
    # 【重要】インデックス列 'X' を明示的に除去
    feature_vars <- setdiff(feature_vars_all, "X")

    for (target_var in target_vars) {
        cat("\n---\nTraining kNN for:", target_var, "on", file, "\n")
        
        df_temp <- df_all[, c(feature_vars, target_var)]
        df_complete <- df_temp[complete.cases(df_temp), ]

        if (nrow(df_complete) < 20) {
            cat("  Skipping due to insufficient data.\n")
            next
        }

        # OOD分割の実行 (FPデータのみ)
        is_fp_data <- grepl("_FP\\.csv$", file) 
        if (is_fp_data) {
            cat("  OOD split based on structural distance...\n")
            split_list <- create_ood_split(df_complete, feature_vars, ood_ratio = 0.1) 
            df_train_cv <- split_list$train
            df_ood_test <- split_list$test 
        } else {
            df_train_cv <- df_complete
            df_ood_test <- NULL
        }
        
        # kNNモデルの訓練
        tune_grid <- expand.grid(k = seq(1, 20, by = 2))
        ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")

        model <- train(
            formula(paste(target_var, "~ .")),
            data = df_train_cv,
            method = "knn",
            metric = "RMSE",
            trControl = ctrl,
            tuneGrid = tune_grid,
            preProcess = c("center", "scale")
        )

        # CV結果の計算
        pred_df <- model$pred
        pred_df <- pred_df[pred_df$k == model$bestTune$k, ]
        
        if (nrow(pred_df) > 0) {
            pred <- pred_df$pred; obs <- pred_df$obs
            R2 <- round(cor(pred, obs)^2, 3)
            RMSE_val <- round(rmse(obs, pred), 3)
            MAE_val <- round(mae(obs, pred), 3)
            RPD_val <- round(sd(obs) / RMSE_val, 3)
            best_k <- model$bestTune$k
            
            result_matrix[paste0("Best_k_", target_var), file] <- best_k
            result_matrix[paste0("R2_", target_var), file] <- R2
            result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
            result_matrix[paste0("MAE_", target_var), file] <- MAE_val
            result_matrix[paste0("RPD_", target_var), file] <- RPD_val
        }
        
        result_matrix[paste0("n_samples_", target_var), file] <- nrow(df_complete)
        result_matrix[paste0("n_features_", target_var), file] <- length(feature_vars)

        # OODテスト評価
        if (is_fp_data && !is.null(df_ood_test) && nrow(df_ood_test) > 0) {
            ood_preds <- predict(model, newdata = df_ood_test)
            ood_obs <- df_ood_test[[target_var]]
            result_matrix[paste0("OOD_R2_", target_var), file] <- round(cor(ood_obs, ood_preds)^2, 3)
            result_matrix[paste0("OOD_RMSE_", target_var), file] <- round(rmse(ood_obs, ood_preds), 3)
        }

        # モデル保存 (ファイル名を 'fixed' に変更)
        model_data_bundle <- list(model = model, training_data = df_train_cv, ood_test_data = df_ood_test)
        rds_file <- paste0("fixed20250702_knn_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".rds")
        saveRDS(model_data_bundle, file = rds_file)
    }
}

# 最終結果の保存 (ファイル名を 'fixed' に変更)
output_file <- paste0("fixed20250702_knn_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\n✅ Fixed kNN summary saved as:", output_file, "\n")


=== Processing: n_base.csv ===

---
Training kNN for: Jsc on n_base.csv 

---
Training kNN for: Voc on n_base.csv 

---
Training kNN for: FF on n_base.csv 

---
Training kNN for: PCEmax on n_base.csv 

=== Processing: n_base_OH.csv ===

---
Training kNN for: Jsc on n_base_OH.csv 


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_ICMA, SMILESsnamep1M_DTSTChy, SMILESsnamep1M_DTTPTP5, SMILESsnamep1M_PBDTTPD, SMILESsnamep1M_PBDTTS3, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2thienyl.2.3dioctylpyrazino.2.3d.pyridazine.P2"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_ICMA, SMILESsnamep1M_DTSTChy, SMILESsnamep1M_DTTPTP5, SMILESsnamep1M_PBDTTPD, SMILESsnamep1M_PBDTTS3, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2thienyl.2.3dioctylpyrazino.2.3d.pyridazine.P2"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_ICMA, SMILESsnamep1M_DTSTChy, SMILESsnamep1M_DTTPTP5, SMILESsnamep1M_PBDTTPD, SMILESsnamep1M_PBDTTS3, SMILESsnamep1M_Poly.N9.fheptadecanyl2.

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamep1M_BDTH7, SMILESsnamep1M_DTST6, SMILESsnamep1M_DTTPTP6, SMILESsnamep1M_PCDTDPP, SMILESsnamep1M_PCDTPX, SMILESsnamep1M_PDTPDTDPP"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamep1M_BDTH7, SMILESsnamep1M_DTST6, SMILESsnamep1M_DTTPTP6, SMILESsnamep1M_PCDTDPP, SMILESsnamep1M_PCDTPX, SMILESsnamep1M_PDTPDTDPP"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamep1M_BDTH7, SMILESsnamep1M_DTST6, SMILESsnamep1M_DTTPTP6, SMILESsnamep1M_PCDTDPP, SMILESsnamep1M_PCDTPX, SMILESsnamep1M_PDTPDTDPP"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_NCMM, SMILESsnamep1M_BDTH11, SMILESsnamep1M

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_PFP, SMILESsnamep1M_DTSC6, SMILESsnamep1M_DTTPTP2, SMILESsnamep1M_DTTPTP3, SMILESsnamep1M_PFBTBDT, SMILESsnamep1M_PTB1, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2thienyl.pyrazino.2.3d.pyridazine.P3"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_PFP, SMILESsnamep1M_DTSC6, SMILESsnamep1M_DTTPTP2, SMILESsnamep1M_DTTPTP3, SMILESsnamep1M_PFBTBDT, SMILESsnamep1M_PTB1, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2thienyl.pyrazino.2.3d.pyridazine.P3"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_PFP, SMILESsnamep1M_DTSC6, SMILESsnamep1M_DTTPTP2, SMILESsnamep1M_DTTPTP3, SMILESsnamep1M_PFBTBDT, SMILESsnamep1M_PTB1, SMIL

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamep1M_DTSDC, SMILESsnamep1M_DTTPTP1, SMILESsnamep1M_PCDTTPP, SMILESsnamep1M_PDPP2FTC12, SMILESsnamep1M_PTB4, SMILESsnamep1M_PTzQT14"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_MOPFP, SMILESsnamenM_TC61BM, SMILESsnamep1M_BDTC6, SMILESsnamep1M_DTSC8, SMILESsnamep1M_PBDTTEHDTHBTffP1, SMILESsnamep1M_PDPP2FTC14"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_MOPFP, SMILESsnamenM_TC61BM, SMILESsnamep1M_BDTC6, SMILESsnamep1M_DTSC8, SMILESsnamep1M_PBDTTEHDTHBTffP1, SMILESsnamep1M_PDPP2FTC14"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_MOPFP, SMILESsnamenM_TC61BM, S

...

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X19, X22, X62, X66, X93, X96, X105, X108, X112, X116, X118, X123, X125, X126, X132, X144, X145, X147, X154, X155, X157, X159, X160, X162, X163, X164, X165, X17.1, X19.1, X22.1, X26.1, X34.1, X36.1, X45.1, X50.1, X62.1, X65.1, X66.1, X75.1, X76.1, X80.1, X81.1, X83.1, X85.1, X88.1, X90.1, X91.1, X96.1, X99.1, X100.1, X101.1, X105.1, X108.1, X109.1, X112.1, X114.1, X115.1, X116.1, X118.1, X120.1, X121.1, X122.1, X123.1, X125.1, X126.1, X128.1, X129.1, X131.1, X132.1, X137.1, X138.1, X141.1, X142.1, X144.1, X145.1, X147.1, X148.1, X149.1, X150.1, X151.1, X153.1, X154.1, X155.1, X156.1, X157.1, X158.1, X159.1, X160.1, X161.1, X162.1, X163.1, X164.1, X165.1, X65.2, X80.2, X83.2, X96.2, X105.2, X120.2, X121.2, X125.2, X131.2, X137.2, X142.2, X144.2, X145.2, X151.2, X156.2, X161.2, X162.2, X163.2, X165.2, X44.3, X47.3, X57.3, X65.3, X66.3, X72.3, X75.3, X79.3, X80.

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X19, X22, X62, X66, X93, X96, X105, X108, X112, X116, X118, X123, X125, X126, X132, X144, X145, X147, X154, X155, X157, X159, X160, X162, X163, X164, X165, X17.1, X19.1, X22.1, X26.1, X34.1, X36.1, X45.1, X50.1, X62.1, X65.1, X66.1, X75.1, X76.1, X80.1, X81.1, X83.1, X85.1, X88.1, X90.1, X91.1, X96.1, X99.1, X100.1, X101.1, X105.1, X108.1, X109.1, X112.1, X114.1, X115.1, X116.1, X118.1, X120.1, X121.1, X122.1, X123.1, X125.1, X126.1, X128.1, X129.1, X131.1, X132.1, X137.1, X138.1, X141.1, X142.1, X144.1, X145.1, X147.1, X148.1, X149.1, X150.1, X151.1, X153.1, X154.1, X155.1, X156.1, X157.1, X158.1, X159.1, X160.1, X161.1, X162.1, X163.1, X164.1, X165.1, X65.2, X80.2, X83.2, X96.2, X105.2, X120.2, X121.2, X125.2, X131.2, X137.2, X142.2, X144.2, X145.2, X151.2, X156.2, X161.2, X162.2, X163.2, X165.2, X44.3, X47.3, X57.3, X65.3, X66.3, X72.3, X75.3, X79.3, X80.

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X19, X22, X62, X66, X93, X96, X105, X108, X112, X116, X118, X123, X125, X126, X132, X144, X145, X147, X154, X155, X157, X159, X160, X162, X163, X164, X165, X17.1, X19.1, X22.1, X26.1, X34.1, X36.1, X45.1, X50.1, X62.1, X65.1, X66.1, X75.1, X76.1, X80.1, X81.1, X83.1, X85.1, X88.1, X90.1, X91.1, X96.1, X99.1, X100.1, X101.1, X105.1, X108.1, X109.1, X112.1, X114.1, X115.1, X116.1, X118.1, X120.1, X121.1, X122.1, X123.1, X125.1, X126.1, X128.1, X129.1, X131.1, X132.1, X137.1, X138.1, X141.1, X142.1, X144.1, X145.1, X147.1, X148.1, X149.1, X150.1, X151.1, X153.1, X154.1, X155.1, X156.1, X157.1, X158.1, X159.1, X160.1, X161.1, X162.1, X163.1, X164.1, X165.1, X65.2, X80.2, X83.2, X96.2, X105.2, X120.2, X121.2, X125.2, X131.2, X137.2, X142.2, X144.2, X145.2, X151.2, X156.2, X161.2, X162.2, X163.2, X165.2, X44.3, X47.3, X57.3, X65.3, X66.3, X72.3, X75.3, X79.3, X80.

Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X19, X22, X62, X66, X93, X96, X105, X108, X112, X116, X118, X123, X125, X126, X132, X144, X145, X147, X154, X155, X157, X159, X160, X162, X163, X164, X165, X17.1, X19.1, X22.1, X26.1, X34.1, X36.1, X45.1, X50.1, X62.1, X65.1, X66.1, X75.1, X76.1, X80.1, X81.1, X83.1, X85.1, X88.1, X90.1, X91.1, X96.1, X99.1, X100.1, X101.1, X105.1, X108.1, X109.1, X112.1, X114.1, X115.1, X116.1, X118.1, X120.1, X121.1, X122.1, X123.1, X125.1, X126.1, X128.1, X129.1, X131.1, X132.1, X137.1, X138.1, X141.1, X142.1, X144.1, X145.1, X147.1, X148.1, X149.1, X150.1, X151.1, X153.1, X154.1, X155.1, X156.1, X157.1, X158.1, X159.1, X160.1, X161.1, X162.1, X163.1, X164.1, X165.1, X65.2, X80.2, X83.2, X96.2, X105.2, X120.2, X121.2, X125.2, X131.2, X137.2, X142.2, X144.2, X145.2, X151.2, X156.2, X161.2, X162.2, X163.2, X165.2, X44.3, X47.3, X57.3, X65.3, X66.3, X72.3, X75.3, X79.3, X80.


<U+2705> Fixed kNN summary saved as: fixed20250702_knn_summary_20251214.csv 
